# SimpleTM — Multivariate Time Series Forecasting

## BTP Project: Comparing SWT vs FFT Tokenization

This notebook runs all 3 model variants:
1. **SimpleTM** — Original (SWT tokenization + Geometric Attention)
2. **SimpleTM_SWT** — SWT-only baseline
3. **SimpleTM_FFT** — FFT tokenization + Geometric Attention

---

## 1. Setup & Install Dependencies

In [1]:
!pip install PyWavelets gdown --quiet

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.9.0+cu126
CUDA available: True
GPU: Tesla T4


In [2]:
# ======================================================================
# Patch for NumPy 2.0 compatibility (np.Inf → np.inf)
# ======================================================================
import subprocess
import os
REPO_DIR = "/kaggle/working/SimpleTM"
# After cloning, patch the file
def patch_numpy_compat(repo_dir):
    """Fix np.Inf → np.inf for NumPy 2.0+ compatibility."""
    tools_path = os.path.join(repo_dir, "utils/tools.py")
    if os.path.exists(tools_path):
        with open(tools_path, 'r') as f:
            content = f.read()
        content = content.replace('np.Inf', 'np.inf')
        with open(tools_path, 'w') as f:
            f.write(content)
        print("✓ Patched utils/tools.py: np.Inf → np.inf")
    else:
        print(f"⚠ {tools_path} not found yet — run this after cloning")

## 2. Clone Repository

In [3]:
import os

# Clone the repo (update URL to your fork if needed)
REPO_URL = "https://github.com/agamswami/SimpleTMG.git"
REPO_DIR = "/kaggle/working/SimpleTM"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
    print("Repository cloned.")
else:
    print("Repository already exists.")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
!ls -la

Cloning into '/kaggle/working/SimpleTM'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 57 (delta 12), reused 57 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (57/57), 7.08 MiB | 27.58 MiB/s, done.
Resolving deltas: 100% (12/12), done.
Repository cloned.
Working directory: /kaggle/working/SimpleTM
total 5728
drwxr-xr-x 10 root root    4096 Mar 10 16:05  .
drwxr-xr-x  4 root root    4096 Mar 10 16:05  ..
-rw-r--r--  1 root root 5093822 Mar 10 16:05  3089_SimpleTM_A_Simple_Baselin.pdf
-rw-r--r--  1 root root  233348 Mar 10 16:05  btp_context.pdf
drwxr-xr-x  2 root root    4096 Mar 10 16:05  data_provider
-rw-r--r--  1 root root     301 Mar 10 16:05  Dockerfile
-rw-r--r--  1 root root     268 Mar 10 16:05  environment.yml
drwxr-xr-x  2 root root    4096 Mar 10 16:05  experiments
drwxr-xr-x  2 root root    4096 Mar 10 16:05  figures
drwxr-xr-x  8 root root    4096 Mar 10 16:0

In [4]:
!sed -i 's/np\.Inf/np.inf/g' /kaggle/working/SimpleTM/utils/tools.py


## 3. Download Dataset

Download the dataset from Google Drive using `gdown`.

In [5]:
!pip install gdown --quiet
import gdown
import zipfile

# Google Drive file ID
FILE_ID = "1hTpUrhe1yEIGa9mCiGxM5rDyzlYKAnyx"
OUTPUT_PATH = "/kaggle/working/dataset.zip"
DATASET_DIR = "./dataset"

if not os.path.exists(DATASET_DIR):
    # Download
    url = f"https://drive.google.com/uc?id={FILE_ID}"
    gdown.download(url, OUTPUT_PATH, quiet=False)
    
    # Extract
    os.makedirs(DATASET_DIR, exist_ok=True)
    if OUTPUT_PATH.endswith('.zip'):
        with zipfile.ZipFile(OUTPUT_PATH, 'r') as zip_ref:
            zip_ref.extractall(DATASET_DIR)
        print(f"Extracted dataset to {DATASET_DIR}")
    else:
        # If it's not a zip, it might be a single file — move it
        import shutil
        shutil.move(OUTPUT_PATH, os.path.join(DATASET_DIR, 'data.csv'))
        print(f"Moved dataset to {DATASET_DIR}")
else:
    print(f"Dataset directory already exists: {DATASET_DIR}")

# List dataset contents
for root, dirs, files in os.walk(DATASET_DIR):
    level = root.replace(DATASET_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files[:10]:
        filepath = os.path.join(root, file)
        size_mb = os.path.getsize(filepath) / (1024*1024)
        print(f'{subindent}{file} ({size_mb:.2f} MB)')

Downloading...
From (original): https://drive.google.com/uc?id=1hTpUrhe1yEIGa9mCiGxM5rDyzlYKAnyx
From (redirected): https://drive.google.com/uc?id=1hTpUrhe1yEIGa9mCiGxM5rDyzlYKAnyx&confirm=t&uuid=2695f96b-5fcd-4f0f-89a3-3a5d3d35c0e2
To: /kaggle/working/dataset.zip
100%|██████████| 171M/171M [00:01<00:00, 97.4MB/s] 


Extracted dataset to ./dataset
dataset/
  SimpleTM_datasets/
    .DS_Store (0.01 MB)
    weather/
      weather.csv (6.90 MB)
    PEMS/
      PEMS07.npz (41.68 MB)
      PEMS04.npz (31.43 MB)
      PEMS03.npz (15.07 MB)
      PEMS08.npz (17.67 MB)
    Solar/
      solar_AL.txt (171.68 MB)
    traffic/
      .DS_Store (0.01 MB)
      traffic.csv (130.16 MB)
    ETT-small/
      ETTh2.csv (2.31 MB)
      ETTh1.csv (2.47 MB)
      ETTm2.csv (9.23 MB)
      ETTm1.csv (9.88 MB)
    electricity/
      electricity.csv (91.15 MB)
      .DS_Store (0.01 MB)
  __MACOSX/
    ._SimpleTM_datasets (0.00 MB)
    SimpleTM_datasets/
      ._electricity (0.00 MB)
      ._weather (0.00 MB)
      ._PEMS (0.00 MB)
      ._ETT-small (0.00 MB)
      ._.DS_Store (0.00 MB)
      ._traffic (0.00 MB)
      ._Solar (0.00 MB)
      weather/
        ._weather.csv (0.00 MB)
      PEMS/
        ._PEMS07.npz (0.00 MB)
        ._PEMS03.npz (0.00 MB)
        ._PEMS08.npz (0.00 MB)
        ._PEMS04.npz (0.00 MB)
      Sol

## 4. Configuration

Set up common experiment parameters. **Adjust `ROOT_PATH` and `DATA_PATH` based on your dataset structure.**

In [6]:
# ======================================================================
# CONFIGURE THESE BASED ON YOUR DATASET
# ======================================================================

# For ETTh1 dataset:
ROOT_PATH = "/kaggle/working/SimpleTM/dataset/SimpleTM_datasets/ETT-small"  # Adjust this path!
DATA_PATH = "ETTm1.csv"             # Adjust this filename!
DATA_TYPE = "ETTm1"                 # Options: ETTh1, ETTh2, ETTm1, ETTm2, custom
ENC_IN = 7                          # Number of variates
DEC_IN = 7
C_OUT = 7

# Common hyperparameters
SEQ_LEN = 96         # Input sequence length
PRED_LEN = 96        # Prediction horizon
D_MODEL = 32         # Token dimension
D_FF = 32            # Feed-forward dimension
E_LAYERS = 1         # Number of encoder layers
M = 3                # Decomposition levels (SWT/FFT bands)
ALPHA = 1.0          # Geometric attention alpha
LR = 0.0001          # Learning rate
BATCH_SIZE = 32      # Batch size
TRAIN_EPOCHS = 10    # Training epochs
PATIENCE = 3         # Early stopping patience

print(f"Dataset: {DATA_TYPE} ({DATA_PATH})")
print(f"Variates: {ENC_IN}, Seq: {SEQ_LEN}, Pred: {PRED_LEN}")
print(f"Model: d_model={D_MODEL}, d_ff={D_FF}, layers={E_LAYERS}")

Dataset: ETTm1 (ETTm1.csv)
Variates: 7, Seq: 96, Pred: 96
Model: d_model=32, d_ff=32, layers=1


## 5. Train Original SimpleTM (SWT + Geometric Attention)

In [7]:
!python -u run.py \
  --is_training 1 \
  --model SimpleTM \
  --model_id {DATA_TYPE}_original \
  --data {DATA_TYPE} \
  --root_path {ROOT_PATH} \
  --data_path {DATA_PATH} \
  --features M \
  --seq_len {SEQ_LEN} \
  --pred_len {PRED_LEN} \
  --e_layers {E_LAYERS} \
  --d_model {D_MODEL} \
  --d_ff {D_FF} \
  --enc_in {ENC_IN} \
  --dec_in {DEC_IN} \
  --c_out {C_OUT} \
  --wv db1 \
  --m {M} \
  --alpha {ALPHA} \
  --learning_rate {LR} \
  --batch_size {BATCH_SIZE} \
  --train_epochs {TRAIN_EPOCHS} \
  --patience {PATIENCE} \
  --num_workers 2 \
  --fix_seed 2025

Args in experiment:
Namespace(is_training=1, model_id='ETTm1_original', model='SimpleTM', data='ETTm1', root_path='/kaggle/working/SimpleTM/dataset/SimpleTM_datasets/ETT-small', data_path='ETTm1.csv', features='M', target='OT', freq='h', checkpoints='./checkpoints/', seq_len=96, label_len=0, pred_len=96, enc_in=7, dec_in=7, c_out=7, n_heads=8, d_layers=1, moving_avg=25, factor=1, distil=True, dropout=0.1, geomattn_dropout=0.5, embed='timeF', activation='gelu', do_predict=False, num_workers=2, itr=1, train_epochs=10, batch_size=32, patience=3, learning_rate=0.0001, des='test', loss='MSE', lradj='type1', pct_start=0.2, use_amp=False, use_gpu=True, gpu=0, use_multi_gpu=False, devices='0,1,2,3', exp_name='MTSF', channel_independence=False, inverse=False, class_strategy='projection', target_root_path='./data/electricity/', target_data_path='electricity.csv', efficient_training=False, use_norm=True, partial_start_index=0, requires_grad=True, wv='db1', m=3, kernel_size=None, alpha=1.0, l1_wei

## 6. Train SimpleTM_SWT (SWT-Only Baseline)

In [8]:
!python -u run.py \
  --is_training 1 \
  --model SimpleTM_SWT \
  --model_id {DATA_TYPE}_SWT \
  --data {DATA_TYPE} \
  --root_path {ROOT_PATH} \
  --data_path {DATA_PATH} \
  --features M \
  --seq_len {SEQ_LEN} \
  --pred_len {PRED_LEN} \
  --e_layers {E_LAYERS} \
  --d_model {D_MODEL} \
  --d_ff {D_FF} \
  --enc_in {ENC_IN} \
  --dec_in {DEC_IN} \
  --c_out {C_OUT} \
  --wv db1 \
  --m {M} \
  --alpha {ALPHA} \
  --learning_rate {LR} \
  --batch_size {BATCH_SIZE} \
  --train_epochs {TRAIN_EPOCHS} \
  --patience {PATIENCE} \
  --num_workers 2 \
  --fix_seed 2025

Args in experiment:
Namespace(is_training=1, model_id='ETTm1_SWT', model='SimpleTM_SWT', data='ETTm1', root_path='/kaggle/working/SimpleTM/dataset/SimpleTM_datasets/ETT-small', data_path='ETTm1.csv', features='M', target='OT', freq='h', checkpoints='./checkpoints/', seq_len=96, label_len=0, pred_len=96, enc_in=7, dec_in=7, c_out=7, n_heads=8, d_layers=1, moving_avg=25, factor=1, distil=True, dropout=0.1, geomattn_dropout=0.5, embed='timeF', activation='gelu', do_predict=False, num_workers=2, itr=1, train_epochs=10, batch_size=32, patience=3, learning_rate=0.0001, des='test', loss='MSE', lradj='type1', pct_start=0.2, use_amp=False, use_gpu=True, gpu=0, use_multi_gpu=False, devices='0,1,2,3', exp_name='MTSF', channel_independence=False, inverse=False, class_strategy='projection', target_root_path='./data/electricity/', target_data_path='electricity.csv', efficient_training=False, use_norm=True, partial_start_index=0, requires_grad=True, wv='db1', m=3, kernel_size=None, alpha=1.0, l1_weig

## 7. Train SimpleTM_FFT (FFT-Only Tokenization)

In [9]:
!python -u run.py \
  --is_training 1 \
  --model SimpleTM_FFT \
  --model_id {DATA_TYPE}_FFT \
  --data {DATA_TYPE} \
  --root_path {ROOT_PATH} \
  --data_path {DATA_PATH} \
  --features M \
  --seq_len {SEQ_LEN} \
  --pred_len {PRED_LEN} \
  --e_layers {E_LAYERS} \
  --d_model {D_MODEL} \
  --d_ff {D_FF} \
  --enc_in {ENC_IN} \
  --dec_in {DEC_IN} \
  --c_out {C_OUT} \
  --wv db1 \
  --m {M} \
  --alpha {ALPHA} \
  --learning_rate {LR} \
  --batch_size {BATCH_SIZE} \
  --train_epochs {TRAIN_EPOCHS} \
  --patience {PATIENCE} \
  --num_workers 2 \
  --fix_seed 2025

Args in experiment:
Namespace(is_training=1, model_id='ETTm1_FFT', model='SimpleTM_FFT', data='ETTm1', root_path='/kaggle/working/SimpleTM/dataset/SimpleTM_datasets/ETT-small', data_path='ETTm1.csv', features='M', target='OT', freq='h', checkpoints='./checkpoints/', seq_len=96, label_len=0, pred_len=96, enc_in=7, dec_in=7, c_out=7, n_heads=8, d_layers=1, moving_avg=25, factor=1, distil=True, dropout=0.1, geomattn_dropout=0.5, embed='timeF', activation='gelu', do_predict=False, num_workers=2, itr=1, train_epochs=10, batch_size=32, patience=3, learning_rate=0.0001, des='test', loss='MSE', lradj='type1', pct_start=0.2, use_amp=False, use_gpu=True, gpu=0, use_multi_gpu=False, devices='0,1,2,3', exp_name='MTSF', channel_independence=False, inverse=False, class_strategy='projection', target_root_path='./data/electricity/', target_data_path='electricity.csv', efficient_training=False, use_norm=True, partial_start_index=0, requires_grad=True, wv='db1', m=3, kernel_size=None, alpha=1.0, l1_weig

## 8. Compare Results

Parse the results file and display a comparison table.

In [10]:
import re
import pandas as pd

results_file = "result_long_term_forecast.txt"

if os.path.exists(results_file):
    with open(results_file, 'r') as f:
        content = f.read()
    
    # Parse results
    entries = content.strip().split('\n\n')
    results = []
    
    for entry in entries:
        lines = entry.strip().split('\n')
        if len(lines) >= 2:
            setting = lines[0].strip()
            metrics_line = lines[1].strip()
            
            # Extract model type from setting
            if '_FFT_' in setting or setting.startswith('FFT'):
                model_type = 'SimpleTM_FFT'
            elif '_SWT_' in setting or setting.startswith('SWT'):
                model_type = 'SimpleTM_SWT'
            else:
                model_type = 'SimpleTM (Original)'
            
            # Parse MSE and MAE
            mse_match = re.search(r'mse:([\d.]+)', metrics_line)
            mae_match = re.search(r'mae:([\d.]+)', metrics_line)
            
            if mse_match and mae_match:
                results.append({
                    'Model': model_type,
                    'Setting': setting[:60] + '...' if len(setting) > 60 else setting,
                    'MSE': float(mse_match.group(1)),
                    'MAE': float(mae_match.group(1)),
                })
    
    if results:
        df = pd.DataFrame(results)
        print("\n" + "="*70)
        print("                    RESULTS COMPARISON")
        print("="*70)
        print(df.to_string(index=False))
        print("="*70)
        
        # Summary by model type
        print("\n--- Average Metrics by Model ---")
        summary = df.groupby('Model')[['MSE', 'MAE']].mean()
        print(summary.to_string())
    else:
        print("No parseable results found.")
else:
    print(f"Results file '{results_file}' not found. Run training first!")


                    RESULTS COMPARISON
              Model                                                         Setting      MSE      MAE
SimpleTM (Original) ETTm1_original_ETTm1_96_96_32_32_1_db1_None_3_1.0_5e-05_0.00... 0.349717 0.379085
       SimpleTM_SWT ETTm1_SWT_ETTm1_96_96_32_32_1_db1_None_3_1.0_5e-05_0.0001_ty... 0.349717 0.379085
       SimpleTM_FFT ETTm1_FFT_ETTm1_96_96_32_32_1_db1_None_3_1.0_5e-05_0.0001_ty... 0.363031 0.382873

--- Average Metrics by Model ---
                          MSE       MAE
Model                                  
SimpleTM (Original)  0.349717  0.379085
SimpleTM_FFT         0.363031  0.382873
SimpleTM_SWT         0.349717  0.379085


## 9. Visualize Predictions

Display saved prediction vs ground-truth plots.

In [11]:
import glob
from IPython.display import Image, display
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

checkpoint_dirs = sorted(glob.glob('./checkpoints/*'))
print(f"Found {len(checkpoint_dirs)} checkpoint directories:")
for d in checkpoint_dirs:
    pdfs = glob.glob(os.path.join(d, '*.pdf'))
    print(f"  {os.path.basename(d)}: {len(pdfs)} visualization PDFs")

# Display first visualization from each model
for d in checkpoint_dirs:
    pdfs = sorted(glob.glob(os.path.join(d, '*.pdf')))
    if pdfs:
        print(f"\n--- {os.path.basename(d)} ---")
        print(f"First visualization: {pdfs[0]}")

Found 3 checkpoint directories:
  ETTm1_FFT_ETTm1_96_96_32_32_1_db1_None_3_1.0_5e-05_0.0001_type1_32_2025_True: 18 visualization PDFs
  ETTm1_SWT_ETTm1_96_96_32_32_1_db1_None_3_1.0_5e-05_0.0001_type1_32_2025_True: 18 visualization PDFs
  ETTm1_original_ETTm1_96_96_32_32_1_db1_None_3_1.0_5e-05_0.0001_type1_32_2025_True: 18 visualization PDFs

--- ETTm1_FFT_ETTm1_96_96_32_32_1_db1_None_3_1.0_5e-05_0.0001_type1_32_2025_True ---
First visualization: ./checkpoints/ETTm1_FFT_ETTm1_96_96_32_32_1_db1_None_3_1.0_5e-05_0.0001_type1_32_2025_True/0.pdf

--- ETTm1_SWT_ETTm1_96_96_32_32_1_db1_None_3_1.0_5e-05_0.0001_type1_32_2025_True ---
First visualization: ./checkpoints/ETTm1_SWT_ETTm1_96_96_32_32_1_db1_None_3_1.0_5e-05_0.0001_type1_32_2025_True/0.pdf

--- ETTm1_original_ETTm1_96_96_32_32_1_db1_None_3_1.0_5e-05_0.0001_type1_32_2025_True ---
First visualization: ./checkpoints/ETTm1_original_ETTm1_96_96_32_32_1_db1_None_3_1.0_5e-05_0.0001_type1_32_2025_True/0.pdf
